# Vision Chat cloud server (Google Colab)
OpenAI-compatible chat (Qwen3-VL, vision, streaming) + image generation (SDXL) for the Vision Chat app.

> **WARNING.** Colab free tier has no guaranteed GPU, sessions end after idle time or a few hours, and Google's terms may restrict some uses (e.g. serving/proxying or tunnelling from notebooks, remote-access use). **Check the current Colab terms/FAQ yourself before using this.** If in doubt use the Kaggle notebook. Use at your own risk.

**Steps:** Runtime -> Change runtime type -> **T4 GPU**. Run all cells top to bottom. First run builds llama.cpp (~10-15 min) and downloads ~15 GB. The last cell prints the URL and API key; keep it running and the tab open. Colab has one 15 GB T4, so the chat and image models are swapped automatically (a few seconds to ~half a minute on switch).

## 1. Config and dependencies

In [ ]:
import os, subprocess, secrets as _sec, sys, shutil
WORK = "/content"
os.makedirs(WORK, exist_ok=True); os.chdir(WORK)

# ---- choices ----
CHAT = "8b"          # "8b" (better, ~6 GB) or "4b" (faster, ~3 GB)
CHAT_CTX = 8192      # context tokens (images use ~300-1500 each)
IMAGE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"   # general-purpose SDXL, fp16
# --------------------

try:
    from google.colab import userdata
    def secret(name):
        try: return userdata.get(name) or ""
        except Exception: return ""
except Exception:
    def secret(name): return ""

API_KEY = secret("API_KEY") or _sec.token_urlsafe(24)
NGROK_TOKEN = secret("NGROK_TOKEN")

print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "NO GPU! Enable a GPU accelerator.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "fastapi", "uvicorn", "httpx", "pydantic>=2", "huggingface_hub", "diffusers==0.35.1",
                "transformers", "accelerate", "safetensors", "sentencepiece"], check=True)
print("deps ok; nvcc:", shutil.which("nvcc") or "NOT FOUND (needed to build llama.cpp)")

## 2. Build llama.cpp `llama-server` with CUDA (cached in this session; ~10-15 min first time)

In [ ]:
import os, subprocess
BIN = f"{WORK}/llama.cpp/build/bin/llama-server"
if not os.path.exists(BIN):
    cuda_home = "/usr/local/cuda" if os.path.exists("/usr/local/cuda") else ""
    env = dict(os.environ)
    if cuda_home: env["PATH"] = cuda_home + "/bin:" + env["PATH"]
    if not os.path.exists(f"{WORK}/llama.cpp"):
        subprocess.run(f"git clone --depth 1 https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp", shell=True, check=True)
    # sm_75 = T4, sm_60 = P100 (Kaggle). Only these archs -> much faster build.
    subprocess.run("cd {0}/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES='60;75' "
                   "-DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF "
                   "&& cmake --build build --config Release -j$(nproc) --target llama-server".format(WORK),
                   shell=True, check=True, env=env)
print("llama-server:", BIN, os.path.exists(BIN))

## 3. Download models from Hugging Face

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
REPOS = {
    "8b": ("prithivMLmods/Qwen3-VL-8B-Instruct-abliterated-v2-GGUF", "Q4_K_M"),
    "4b": ("noctrex/Huihui-Qwen3-VL-4B-Instruct-abliterated-GGUF", "Q4_K_M"),
}
repo, quant = REPOS[CHAT]
files = HfApi().list_repo_files(repo)
print("\n".join(files))
ggufs = [f for f in files if f.endswith(".gguf")]
main = [f for f in ggufs if "mmproj" not in f.lower() and quant.lower() in f.lower()]
mm = [f for f in ggufs if "mmproj" in f.lower()]
mm = sorted(mm, key=lambda f: 0 if "q8_0" in f.lower() else (1 if "f16" in f.lower() else 2))
assert main and mm, "could not find model/mmproj files, see list above and set names by hand"
CHAT_MODEL = hf_hub_download(repo, main[0], local_dir=WORK + "/models")
CHAT_MMPROJ = hf_hub_download(repo, mm[0], local_dir=WORK + "/models")
print("chat:", CHAT_MODEL, "\nmmproj:", CHAT_MMPROJ)

from huggingface_hub import snapshot_download
# fp16 SDXL only (skip the fp32 / onnx / openvino copies) + fp16-safe VAE
snapshot_download(IMAGE_MODEL, allow_patterns=["*.json", "*.txt", "*fp16.safetensors"],
                  ignore_patterns=["*.onnx*", "*openvino*", "sd_xl_*", "*_pruned*", "*.ckpt"])
snapshot_download("madebyollin/sdxl-vae-fp16-fix", allow_patterns=["*.json", "*.safetensors"])
print("image model cached")

## 4. Start the server

In [ ]:
import os, subprocess, time, urllib.request
for p in ("server.py",):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/houlashdw/Iaa/main/cloud/" + p, f"{WORK}/" + p)
env = dict(os.environ, API_KEY=API_KEY, PORT="8000", LLAMA_SERVER_BIN=BIN,
           CHAT_MODEL=CHAT_MODEL, CHAT_MMPROJ=CHAT_MMPROJ, CHAT_CTX=str(CHAT_CTX),
           IMAGE_MODEL=IMAGE_MODEL, LOG_DIR=WORK, IMAGE_KEEP_CPU="0",
           HF_HUB_ENABLE_HF_TRANSFER="0")
server = subprocess.Popen([sys.executable, f"{WORK}/server.py"], env=env,
                          stdout=open(f"{WORK}/server.log", "ab"), stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        print(urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3).read().decode()); break
    except Exception:
        time.sleep(2)
else:
    print(open(f"{WORK}/server.log").read()[-3000:])
print("(the chat model loads in the background, first request may wait up to a minute)")

## 5. Public tunnel (cloudflared quick tunnel; ngrok if `NGROK_TOKEN` is set and cloudflared fails)

In [ ]:
import re, subprocess, time, os, urllib.request
CF = WORK + "/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
PUBLIC_URL = None
tun = subprocess.Popen([CF, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
t0 = time.time()
while time.time() - t0 < 60 and PUBLIC_URL is None:
    line = tun.stdout.readline()
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if m: PUBLIC_URL = m.group(0)
if PUBLIC_URL is None and NGROK_TOKEN:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    PUBLIC_URL = ngrok.connect(8000, "http").public_url
if PUBLIC_URL is None:
    print("TUNNEL FAILED. Check that Internet is ON; add NGROK_TOKEN secret as fallback.")
else:
    print("=" * 60)
    print("  SERVER URL :", PUBLIC_URL)
    print("  API KEY    :", API_KEY)
    print("=" * 60)
    print("In the app: Base URL = URL above (no /v1 needed unless the app asks), key = API KEY.")
    print("Test: curl -H 'Authorization: Bearer KEY' URL/v1/models")

## 6. Keep alive

- Colab free disconnects after ~90 min idle or ~12 h total, sooner under heavy use; keep the tab open and the last cell running.
- If no GPU is available, try later. Do not try to bypass limits.

In [ ]:
# Keep this cell running. Interrupt (stop button) to shut everything down.
import time, urllib.request
try:
    while True:
        try:
            urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).read()
            ok = "ok"
        except Exception as e:
            ok = f"SERVER DOWN: {e}"
        print(time.strftime("%H:%M:%S"), ok, "|", PUBLIC_URL, flush=True)
        time.sleep(300)
except KeyboardInterrupt:
    server.terminate(); tun.terminate(); print("stopped")